In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your real folder uses that spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HIQ_J.xpt"
csv_file = DATA_DIR / "HIQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2017-2018 HIQ_J XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HIQ011",
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA",
    "HIQ260",
    "HIQ105",
    "HIQ270",
    "HIQ210"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HIQ_J columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9254:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 16:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,254 rows x 16 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached HIQ_J.xpt:
#
#     character fields        = NONE
#     genuine decimal fields  = NONE
#
#
# All variables contain:
#
#     respondent identifier
#     categorical insurance codes
#
#
# Therefore all 16 fields can safely become nullable
# Int64 AFTER proving every populated value is
# mathematically whole-number.
#
#
# IMPORTANT:
#
# This conclusion applies ONLY to HIQ_J.
#
# It does NOT change your general NHANES rule:
#
#     genuine Dietary decimals
#     Lab continuous measurements
#     Examination continuous measurements
#     sample weights
#
# remain exactly AS IS.
#

STRING_COLUMNS = []

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 16:

    raise ValueError(
        "Expected exactly 16 whole-number/code fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Character fields:",
    STRING_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# Attached HIQ_J contains NONE.
#
# Therefore NO tiny -> zero repair is required.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 16 HIQ_J variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


ordinary_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)

print(
    "Ordinary numeric zeros:",
    f"{ordinary_zero_total:,}"
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HIQ_J."
    )


if ordinary_zero_total != 0:

    print(
        ordinary_zero_counts[
            ordinary_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary numeric zeros "
        "found in HIQ_J."
    )


print(
    "PASS: HIQ_J contains 0 tiny artifacts "
    "and 0 ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# No numerical repair is necessary.
#
# Therefore the original XPT becomes the exact
# numerical fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero repair required."
)


# ============================================================
# 7. DETECT GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE converting anything to Int64.
#
# np.round() is used ONLY diagnostically.
#
# It NEVER modifies dataframe values.
#

print(
    "\n--- FRACTIONAL VALUE DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = count


    print(
        f"{col}: "
        f"{count:,} fractional observations"
    )


    if count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal fields found.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal/fractional "
    "fields found in HIQ_J."
)


# ============================================================
# 8. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93703:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 93703-102956 validated."
)


# ============================================================
# 9. PROVE ALL 16 VARIABLES ARE WHOLE-NUMBER
# ============================================================

whole_number_failures = []


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        whole_number_failures.append(
            col
        )


if whole_number_failures:

    raise ValueError(
        "Unexpected fractional values found in:\n"
        + str(
            whole_number_failures
        )
    )


print(
    "PASS: All 16 HIQ_J variables are "
    "mathematically whole-number."
)


# ============================================================
# 10. CONVERT ALL 16 FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() is used here.
#
# We have already proven every value is whole-number.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All 16 fields converted to nullable Int64 "
    "without rounding."
)


# ============================================================
# 11. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column}"
    )


# ============================================================
# 12. CDC CHECK - HIQ011
# ============================================================

check_frequency(
    "HIQ011",
    {
        1: 8157,
        2: 1072,
        7: 7,
        9: 18
    },
    0
)


# ============================================================
# 13. CDC CHECK - HIQ031A
# ============================================================

check_frequency(
    "HIQ031A",
    {
        14: 4188,
        77: 3,
        99: 63
    },
    5000
)


# ============================================================
# 14. CDC CHECK - HIQ031B
# ============================================================

check_frequency(
    "HIQ031B",
    {
        15: 1582
    },
    7672
)


# ============================================================
# 15. CDC CHECK - HIQ031C
# ============================================================

check_frequency(
    "HIQ031C",
    {
        16: 58
    },
    9196
)


# ============================================================
# 16. CDC CHECK - HIQ031D
# ============================================================

check_frequency(
    "HIQ031D",
    {
        17: 2527
    },
    6727
)


# ============================================================
# 17. CDC CHECK - HIQ031E
# ============================================================

check_frequency(
    "HIQ031E",
    {
        18: 93
    },
    9161
)


# ============================================================
# 18. CDC CHECK - HIQ031F
# ============================================================

check_frequency(
    "HIQ031F",
    {
        19: 295
    },
    8959
)


# ============================================================
# 19. CDC CHECK - HIQ031H
# ============================================================

check_frequency(
    "HIQ031H",
    {
        21: 533
    },
    8721
)


# ============================================================
# 20. CDC CHECK - HIQ031I
# ============================================================

check_frequency(
    "HIQ031I",
    {
        22: 301
    },
    8953
)


# ============================================================
# 21. CDC CHECK - HIQ031J
# ============================================================

check_frequency(
    "HIQ031J",
    {
        23: 708
    },
    8546
)


# ============================================================
# 22. CDC CHECK - HIQ031AA
# ============================================================

#
# No coverage of any type.
#
# Attachment / CDC:
#
#     40 = 1 record
#

check_frequency(
    "HIQ031AA",
    {
        40: 1
    },
    9253
)


# ============================================================
# 23. HEALTH-INSURANCE COVERAGE BLOCK VALIDATION
# ============================================================

#
# HIQ031A-J/AA is the multi-response insurance
# coverage block.
#
# In the released attachment:
#
#     HIQ011=1                  = 8,157
#     any HIQ031* populated     = 8,157
#
# with zero row-level mismatches.
#

COVERAGE_COLUMNS = [
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA"
]


hiq011_yes = (
    df["HIQ011"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


coverage_present = (
    df[COVERAGE_COLUMNS]
    .notna()
    .any(axis=1)
    .to_numpy(
        dtype=bool
    )
)


coverage_mismatch = int(
    np.sum(
        hiq011_yes
        !=
        coverage_present
    )
)


print(
    "\n--- HIQ011 / HIQ031 COVERAGE BLOCK ---"
)

print(
    "HIQ011=1:",
    int(
        hiq011_yes.sum()
    )
)

print(
    "Any HIQ031* populated:",
    int(
        coverage_present.sum()
    )
)

print(
    "Row-level mismatches:",
    coverage_mismatch
)


if int(
    hiq011_yes.sum()
) != 8157:

    raise ValueError(
        "Unexpected HIQ011=1 count."
    )


if int(
    coverage_present.sum()
) != 8157:

    raise ValueError(
        "Unexpected coverage-block participant count."
    )


if coverage_mismatch != 0:

    raise ValueError(
        "HIQ031 insurance block does not align "
        "with HIQ011=1."
    )


print(
    "PASS: HIQ011/HIQ031 coverage block validated."
)


# ============================================================
# 24. CHECK HIQ031AA NO-COVERAGE RECORD
# ============================================================

#
# Exactly one insured-response record has:
#
#     HIQ031AA = 40
#
# meaning none of the listed coverage types.
#

hiq031aa_count = int(
    df["HIQ031AA"]
    .eq(40)
    .sum()
)


if hiq031aa_count != 1:

    raise ValueError(
        "Unexpected HIQ031AA=40 count."
    )


hiq031aa_other_coverage = int(
    (
        df["HIQ031AA"].eq(40)
        &
        df[
            [
                "HIQ031A",
                "HIQ031B",
                "HIQ031C",
                "HIQ031D",
                "HIQ031E",
                "HIQ031F",
                "HIQ031H",
                "HIQ031I",
                "HIQ031J"
            ]
        ]
        .notna()
        .any(axis=1)
    )
    .sum()
)


if hiq031aa_other_coverage != 0:

    raise ValueError(
        "HIQ031AA=40 overlaps another coverage type."
    )


print(
    "PASS: HIQ031AA no-coverage record "
    "does not overlap other coverage types."
)


# ============================================================
# 25. CDC CHECK - HIQ260
# ============================================================

#
# Medicare follow-up.
#
# Age is required to independently recreate
# all HIQ260 routing.
#
# Age is NOT stored in HIQ_J.
#
# Therefore we validate the released frequency
# without inventing an age-based routing check.
#

check_frequency(
    "HIQ260",
    {
        1: 77,
        2: 87,
        7: 3,
        9: 5
    },
    9082
)


print(
    "NOTE: HIQ260 age-based eligibility is not "
    "reconstructed because age is not present in HIQ_J."
)


# ============================================================
# 26. CDC CHECK - HIQ105
# ============================================================

#
# Insurance card available.
#
# This also interacts with Medicare/card routing.
#
# We validate exact frequencies but do not impose
# a false simplified routing condition.
#

check_frequency(
    "HIQ105",
    {
        1: 1081,
        2: 60
    },
    8113
)


print(
    "NOTE: No simplified HIQ105 routing assertion "
    "is imposed because the complete age/card logic "
    "is not contained in HIQ_J."
)


# ============================================================
# 27. CDC CHECK - HIQ270
# ============================================================

check_frequency(
    "HIQ270",
    {
        1: 7678,
        2: 379,
        7: 4,
        9: 110
    },
    1083
)


# ============================================================
# 28. CDC HIQ269 -> HIQ270 ROUTING
# ============================================================

#
# CDC HIQ269 check item:
#
# Continue when:
#
#   (HIQ011 = 1 AND HIQ031AA != 40)
#       OR
#   HIQ260 = 1
#
#
# Otherwise end section.
#
# This condition can be reconstructed directly
# from HIQ_J without needing age.
#

no_coverage = (
    df["HIQ031AA"]
    .eq(40)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


insured = (
    df["HIQ011"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


medicare_yes = (
    df["HIQ260"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


hiq270_expected = (
    (
        insured
        &
        ~no_coverage
    )
    |
    medicare_yes
)


hiq270_present = (
    df["HIQ270"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


hiq270_routing_mismatch = int(
    np.sum(
        hiq270_expected
        !=
        hiq270_present
    )
)


print(
    "\n--- HIQ269 / HIQ270 ROUTING ---"
)

print(
    "Expected eligible:",
    int(
        hiq270_expected.sum()
    )
)

print(
    "HIQ270 populated:",
    int(
        hiq270_present.sum()
    )
)

print(
    "Routing mismatches:",
    hiq270_routing_mismatch
)


if int(
    hiq270_expected.sum()
) != 8171:

    raise ValueError(
        "Unexpected HIQ270 eligibility count."
    )


if int(
    hiq270_present.sum()
) != 8171:

    raise ValueError(
        "Unexpected HIQ270 populated count."
    )


if hiq270_routing_mismatch != 0:

    raise ValueError(
        "HIQ270 does not match HIQ269 routing."
    )


print(
    "PASS: HIQ269/HIQ270 routing validated exactly."
)


# ============================================================
# 29. CDC CHECK - HIQ210
# ============================================================

check_frequency(
    "HIQ210",
    {
        1: 561,
        2: 7591,
        7: 1,
        9: 18
    },
    1083
)


# ============================================================
# 30. HIQ210 AVAILABILITY CHECK
# ============================================================

#
# HIQ210 follows the same eligible section flow
# in this released file.
#
# Attachment:
#
#     HIQ270 nonmissing = 8,171
#     HIQ210 nonmissing = 8,171
#
# Their missing/populated masks match exactly.
#

hiq210_present = (
    df["HIQ210"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


hiq210_vs_270_mismatch = int(
    np.sum(
        hiq210_present
        !=
        hiq270_present
    )
)


print(
    "\n--- HIQ270 / HIQ210 AVAILABILITY ---"
)

print(
    "HIQ270 populated:",
    int(
        hiq270_present.sum()
    )
)

print(
    "HIQ210 populated:",
    int(
        hiq210_present.sum()
    )
)

print(
    "Availability mismatches:",
    hiq210_vs_270_mismatch
)


if int(
    hiq210_present.sum()
) != 8171:

    raise ValueError(
        "Unexpected HIQ210 populated count."
    )


if hiq210_vs_270_mismatch != 0:

    raise ValueError(
        "HIQ210 availability differs from HIQ270."
    )


print(
    "PASS: HIQ270/HIQ210 availability masks match."
)


# ============================================================
# 31. VALID RESPONSE CODE CHECK
# ============================================================

VALID_CODES = {

    "HIQ011": {
        1, 2, 7, 9
    },

    "HIQ031A": {
        14, 77, 99
    },

    "HIQ031B": {
        15
    },

    "HIQ031C": {
        16
    },

    "HIQ031D": {
        17
    },

    "HIQ031E": {
        18
    },

    "HIQ031F": {
        19
    },

    "HIQ031H": {
        21
    },

    "HIQ031I": {
        22
    },

    "HIQ031J": {
        23
    },

    "HIQ031AA": {
        40
    },

    "HIQ260": {
        1, 2, 7, 9
    },

    "HIQ105": {
        1, 2
    },

    "HIQ270": {
        1, 2, 7, 9
    },

    "HIQ210": {
        1, 2, 7, 9
    }
}


print(
    "\n--- VALID RESPONSE CODE CHECK ---"
)


for col, allowed_codes in VALID_CODES.items():

    observed_codes = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed_codes
        -
        allowed_codes
    )


    print(
        f"{col}:",
        sorted(
            observed_codes
        )
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


print(
    "PASS: No unexpected HIQ response codes."
)


# ============================================================
# 32. FINAL ZERO CHECK BEFORE EXPORT
# ============================================================

#
# Original XPT:
#
#     tiny zeros      = 0
#     ordinary zeros  = 0
#
# Therefore processing must not create any zeros.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK BEFORE EXPORT ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected numeric zeros created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No numeric zeros created."
)


# ============================================================
# 33. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HIQ_J required NO numerical repairs.
#
# Therefore every value must still be numerically
# identical to source XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 16 variables remain numerically "
    "identical to source XPT."
)


# ============================================================
# 34. STORE MISSING COUNTS BEFORE EXPORT
# ============================================================

expected_missing_counts = {
    col: int(
        expected_df[col]
        .isna()
        .sum()
    )
    for col in EXPECTED_COLUMNS
}


# ============================================================
# 35. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# DO NOT USE:
#
#     df.round(...)
#
# DO NOT USE:
#
#     float_format=
#
#
# HIQ_J has no genuine decimals.
#
# But using this export approach remains consistent
# with your other NHANES files where decimal values
# must stay exactly AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 36. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 37. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 16 HIQ_J variables."
)


# ============================================================
# 38. MISSING-VALUE FIDELITY
# ============================================================

missing_position_mismatches = 0

missing_count_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_position_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    actual_missing_count = int(
        roundtrip_df[col]
        .isna()
        .sum()
    )


    if (
        actual_missing_count
        !=
        expected_missing_counts[col]
    ):

        missing_count_failures.append(
            col
        )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatches
)


if missing_position_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


if missing_count_failures:

    raise ValueError(
        "Missing counts changed in:\n"
        + str(
            missing_count_failures
        )
    )


print(
    "PASS: Missing positions/counts preserved exactly."
)


# ============================================================
# 39. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 40. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 41. RAW CSV .0 CHECK
# ============================================================

#
# HIQ_J contains no genuine decimal columns.
#
# Therefore it is safe to check all 16 fields for
# unnecessary ".0".
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- RAW CSV .0 CHECK ---"
)

print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HIQ_J field."
)


# ============================================================
# 42. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected numeric zeros found "
        "in final CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 43. FINAL KEY FREQUENCY CHECKS
# ============================================================

FINAL_FREQUENCIES = {

    "HIQ011": {
        1: 8157,
        2: 1072,
        7: 7,
        9: 18
    },

    "HIQ031A": {
        14: 4188,
        77: 3,
        99: 63
    },

    "HIQ031B": {
        15: 1582
    },

    "HIQ031C": {
        16: 58
    },

    "HIQ031D": {
        17: 2527
    },

    "HIQ031E": {
        18: 93
    },

    "HIQ031F": {
        19: 295
    },

    "HIQ031H": {
        21: 533
    },

    "HIQ031I": {
        22: 301
    },

    "HIQ031J": {
        23: 708
    },

    "HIQ031AA": {
        40: 1
    },

    "HIQ260": {
        1: 77,
        2: 87,
        7: 3,
        9: 5
    },

    "HIQ105": {
        1: 1081,
        2: 60
    },

    "HIQ270": {
        1: 7678,
        2: 379,
        7: 4,
        9: 110
    },

    "HIQ210": {
        1: 561,
        2: 7591,
        7: 1,
        9: 18
    }
}


print(
    "\n--- FINAL FREQUENCY CHECKS ---"
)


for col, expected_freq in FINAL_FREQUENCIES.items():

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    actual_freq = (
        values
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )


    if actual_freq != expected_freq:

        raise ValueError(
            f"Final {col} frequencies changed.\n\n"
            f"Expected:\n{expected_freq}\n\n"
            f"Actual:\n{actual_freq}"
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 44. FINAL COVERAGE-BLOCK CHECK
# ============================================================

final_insured = (
    pd.to_numeric(
        raw_csv_df["HIQ011"],
        errors="coerce"
    )
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_coverage_present = (
    raw_csv_df[COVERAGE_COLUMNS]
    .ne("")
    .any(axis=1)
    .to_numpy(
        dtype=bool
    )
)


final_coverage_mismatch = int(
    np.sum(
        final_insured
        !=
        final_coverage_present
    )
)


if final_coverage_mismatch != 0:

    raise ValueError(
        "Final HIQ011/HIQ031 relationship changed."
    )


print(
    "PASS: Final HIQ011/HIQ031 coverage block preserved."
)


# ============================================================
# 45. FINAL HIQ269 / HIQ270 ROUTING CHECK
# ============================================================

final_hiq011 = pd.to_numeric(
    raw_csv_df["HIQ011"],
    errors="coerce"
)


final_hiq031aa = pd.to_numeric(
    raw_csv_df["HIQ031AA"]
    .where(
        raw_csv_df["HIQ031AA"] != "",
        np.nan
    ),
    errors="coerce"
)


final_hiq260 = pd.to_numeric(
    raw_csv_df["HIQ260"]
    .where(
        raw_csv_df["HIQ260"] != "",
        np.nan
    ),
    errors="coerce"
)


final_no_coverage = (
    final_hiq031aa
    .eq(40)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_insured = (
    final_hiq011
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_medicare_yes = (
    final_hiq260
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_eligible = (
    (
        final_insured
        &
        ~final_no_coverage
    )
    |
    final_medicare_yes
)


final_hiq270_present = (
    raw_csv_df["HIQ270"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_hiq210_present = (
    raw_csv_df["HIQ210"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_270_mismatch = int(
    np.sum(
        final_eligible
        !=
        final_hiq270_present
    )
)


final_210_mismatch = int(
    np.sum(
        final_eligible
        !=
        final_hiq210_present
    )
)


print(
    "\n--- FINAL HIQ ROUTING CHECK ---"
)

print(
    "Expected eligible:",
    int(
        final_eligible.sum()
    )
)

print(
    "HIQ270 populated:",
    int(
        final_hiq270_present.sum()
    )
)

print(
    "HIQ210 populated:",
    int(
        final_hiq210_present.sum()
    )
)

print(
    "HIQ270 mismatches:",
    final_270_mismatch
)

print(
    "HIQ210 mismatches:",
    final_210_mismatch
)


if int(
    final_eligible.sum()
) != 8171:

    raise ValueError(
        "Final HIQ269 eligibility count changed."
    )


if final_270_mismatch != 0:

    raise ValueError(
        "Final HIQ270 routing changed."
    )


if final_210_mismatch != 0:

    raise ValueError(
        "Final HIQ210 routing changed."
    )


print(
    "PASS: Final HIQ269/HIQ270/HIQ210 "
    "routing preserved."
)


# ============================================================
# 46. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 47. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93703:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 48. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "HIQ_J CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2017-2018"
)


print(
    "Component: Questionnaire - Health Insurance"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny XPORT artifacts:",
    tiny_total
)


print(
    "Original ordinary numeric zeros:",
    ordinary_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Genuine decimal variables:",
    fractional_columns
)


print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "HIQ011 insured:",
    f"{int(df['HIQ011'].eq(1).sum()):,}"
)


print(
    "Any HIQ031 coverage response:",
    f"{int(coverage_present.sum()):,}"
)


print(
    "Coverage-block mismatches:",
    coverage_mismatch
)


print(
    "HIQ260 Medicare yes:",
    int(
        df["HIQ260"]
        .eq(1)
        .sum()
    )
)


print(
    "HIQ270 eligible/populated:",
    f"{int(hiq270_expected.sum()):,}"
)


print(
    "HIQ270 routing mismatches:",
    hiq270_routing_mismatch
)


print(
    "HIQ210 availability mismatches:",
    hiq210_vs_270_mismatch
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "HIQ_J requires NO tiny-zero restoration."
)


print(
    "All 16 fields are proven whole-number "
    "insurance IDs/codes."
)


print(
    "All 16 can therefore be converted to "
    "nullable Int64 without rounding."
)


print(
    "No df.round() and no float_format are used."
)


print(
    "Age-dependent HIQ260/HIQ105 routing is not "
    "artificially reconstructed because age is "
    "not contained in HIQ_J itself."
)


print(
    "Genuine Dietary and all other genuine "
    "continuous decimal variables remain exactly AS IS."
)

NHANES 2017-2018 HIQ_J XPT -> CSV
Rows: 9,254
Columns: 16
PASS: Structure = 9,254 rows x 16 columns.

--- FIELD TYPE PLAN ---
Character fields: []
Genuine decimal fields: []
Whole-number/code fields: 16

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Variables containing tiny artifacts: 0
Ordinary numeric zeros: 0
PASS: HIQ_J contains 0 tiny artifacts and 0 ordinary numeric zeros.

PASS: No tiny-to-zero repair required.

--- FRACTIONAL VALUE DETECTION ---
SEQN: 0 fractional observations
HIQ011: 0 fractional observations
HIQ031A: 0 fractional observations
HIQ031B: 0 fractional observations
HIQ031C: 0 fractional observations
HIQ031D: 0 fractional observations
HIQ031E: 0 fractional observations
HIQ031F: 0 fractional observations
HIQ031H: 0 fractional observations
HIQ031I: 0 fractional observations
HIQ031J: 0 fractional observations
HIQ031AA: 0 fractional observations
HIQ260: 0 fractional observations
HIQ105: 0 fractional observations
HIQ270: 0 fractional observations
HIQ210: 0 fr